In [2]:
import altair as alt
import pandas as pd

url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

cols = ['datetime','city','state','country','shape',
        'duration_seconds','duration_hm','comments',
        'date_posted','latitude','longitude']
df = pd.read_csv(url, names=cols, low_memory=False)

alt.data_transformers.disable_max_rows()


df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = df.dropna(subset=['datetime','shape'])
df['year'] = df['datetime'].dt.year
df = df[df['year'] >= 1950]

shape_counts = df.groupby('shape').size().reset_index(name='count')

chart1 = alt.Chart(shape_counts).mark_bar().encode(
    x=alt.X('count:Q', title='Number of Sightings'),
    y=alt.Y('shape:N', sort='-x', title='UFO Shape'),
    color=alt.Color('count:Q', scale=alt.Scale(scheme='viridis'), title='Sightings'),
    tooltip=['shape:N', 'count:Q']
).properties(
    width=500, height=400,
    title='UFO Sightings by Shape'
)

chart1.save('chart1.json')

top_shapes = df['shape'].value_counts().head(8).index.tolist()
year_shape = df[df['shape'].isin(top_shapes)].groupby(['year','shape']).size().reset_index(name='count')

selection = alt.selection_point(fields=['shape'], bind='legend')

chart2 = alt.Chart(year_shape).mark_line(point=True).encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('count:Q', title='Sightings'),
    color=alt.Color('shape:N', title='Shape'),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.1)),
    tooltip=['year:O', 'shape:N', 'count:Q']
).add_params(selection).properties(
    width=600, height=400,
    title='UFO Sightings Over Time (click legend to filter)'
)

chart2.save('chart2.json')

chart1 | chart2

alt.HConcatChart(...)